# CS-323 AI CEP: Intelligent Vacuum-Cleaning Robot

| Name | Roll number |
|---|---|
| Muhammad Ahmed Qazi | CS-24045 |
| Mujtaba Jawaid Rao | CS-24047 |
| Huzaifa Hanif | CS-24039 |
| Muhammad Ashar Khan | CS-24035 |

The robot plans **when** to clean each dirty cell with a CSP (time slots, precedence,
latest-slot and slot-exclusivity constraints) and **how** to get there with graph search
(UCS and A\*), then executes the schedule starting and ending at the charging station.

*This notebook is generated from the team repository by `tools/build_notebook.py`; run it
top to bottom ("Run All").*

## 0. Configuration: input files

All input is read from files; nothing about the environment or the schedule is hardcoded
(spec §2, §8.1). Change the file names below to run the notebook on other files. Files are
looked up in the notebook's own folder first, then in `data/` and `../data/` (the layout of
the team repository).

In [ ]:
from pathlib import Path

# Input files (spec §8.1): change these names to run other files.
ENV_FILE = "Environment.txt"
SCHEDULE_FILE = "Schedule.txt"
SAMPLE_ENV_FILE = "Environment_Sample.txt"
SAMPLE_SCHEDULE_FILE = "Schedule_Sample.txt"
ERROR_ENV_FILE = "Environment_Error.txt"
ERROR_SCHEDULE_FILE = "Schedule_Error.txt"

# Notebook folder first (submission), then the repository layout.
SEARCH_DIRS = [Path.cwd(), Path.cwd() / "data", Path.cwd().parent / "data"]


def find_file(name):
    """Return the first existing ``name`` in SEARCH_DIRS.

    Args:
        name: File name.

    Returns:
        Path to the file.

    Raises:
        FileNotFoundError: If no folder has it.
    """
    for folder in SEARCH_DIRS:
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f"{name} not found in {[str(d) for d in SEARCH_DIRS]}")

## 1. Environment loading

`Environment.txt` (spec §5.2) is parsed into a frozen `Grid` dataclass: dimensions, a
`frozenset` of obstacle cells, the dirty cells labelled `D1…Dn` in file order, and the
charging station. `Schedule.txt` (spec §6.3) is parsed into a frozen `Constraints`
dataclass. Slots are stored as integers (shown as `T<n>`) so that `T10` sorts after `T9`.

**Design decisions** (team choices where the spec is silent, see `docs/decisions.md`):
- DEC-06: frozen dataclasses — no component can change the shared grid; integration keeps
  its own record of cleaned cells.
- DEC-07: every input problem raises one `InputError` whose message names the file, the
  line and the bad value; the first problem found is reported.
- DEC-08 – DEC-12: tolerant whitespace, empty tuple line allowed for a count of 0, one tuple
  group per line, duplicates/overlaps/out-of-bounds cells rejected, slots written `T3` or `3`.
  Reachability is *not* checked by the parser: an unreachable cell is a runtime failure the
  integration must detect and report (§7.1 item 6).

In [ ]:
"""Environment and schedule loading, grid representation, validation. Owner: Qazi.

Spec §5.2 (Environment Input File):
    "Use the following format, with one item or group of items per line:
    1. Grid dimensions as rows,columns.
    2. Number of obstacles.
    3. Coordinates of all obstacles, represented as (row,column) tuples.
    4. Number of dirty locations.
    5. Coordinates of all dirty locations, represented as (row,column) tuples.
    6. Coordinate of the charging station, represented as a (row,column) tuple.
    Coordinates are zero-based. The first dirty location listed is D1, the
    second is D2, and so on."

Spec §6.3 (Scheduling Input File):
    "1. Number of available time slots.
    2. Number of precedence constraints.
    3. One pair (task_before,task_after) per line for each precedence constraint.
    4. Number of latest-slot constraints.
    5. One pair (task,latest_slot) per line for each latest-slot constraint.
    Slot exclusivity applies automatically to all tasks and does not need to be
    listed in the file."

Spec §6.4:
    "The number of available time slots must be at least the number of dirty
    locations." / "Task labels used in the scheduling constraints must
    correspond to the dirty locations defined by Environment.txt."

Internal types (issue #3, DEC-06; also documented in AGENTS.md). All are team
choices where the spec is silent, except ``State`` (spec §5.3):
    - ``State``: ``(row, col)`` tuple of ints, zero-based (REQ-ENV-03).
    - ``Label``: task label ``"D1"`` … ``"Dn"`` in file order (REQ-ENV-04).
    - ``Slot``: int ``1`` … ``n``, displayed as ``T<n>``; ints because string
      order breaks at ten slots (``"T10" < "T9"``).
    - ``ACTIONS``: ``UP, DOWN, LEFT, RIGHT`` with their row/column deltas.
    - ``ALGORITHMS``: ``"ucs"``, ``"astar"`` (DEC-01).
    - ``Grid``, ``Constraints``: frozen dataclasses below.
Environment parsing: issue #4 (rules DEC-07 to DEC-11, DEC-13). Schedule parsing:
issue #5 (rules DEC-07 to DEC-10, DEC-12, DEC-13).
"""

import re
from collections.abc import Mapping
from dataclasses import dataclass
from pathlib import Path
from types import MappingProxyType

State = tuple[int, int]
Label = str
Slot = int

# Order matters: get_successors yields moves in this order, which fixes how
# UCS/A* break ties between equal-priority nodes. Spec order (§2, §5.3), DEC-06.
ACTIONS: tuple[tuple[str, tuple[int, int]], ...] = (
    ("UP", (-1, 0)),
    ("DOWN", (1, 0)),
    ("LEFT", (0, -1)),
    ("RIGHT", (0, 1)),
)

# search() raises ValueError for any other name, so a typo is never mistaken
# for "no path exists" (DEC-06).
ALGORITHMS: tuple[str, ...] = ("ucs", "astar")


@dataclass(frozen=True)
class Grid:
    """Environment loaded from Environment.txt (spec §5.3, REQ-ENV-06).

    Frozen so that no component can alter the shared environment; integration
    keeps its own record of cleaned cells (REQ-INT-03). Dirty and charger cells
    are passable: only out-of-bounds and obstacle cells block a move.

    Attributes:
        rows: Number of rows (file line 1).
        cols: Number of columns (file line 1).
        obstacles: Obstacle cells (file line 3); a frozenset for O(1) lookup in
            get_successors.
        dirty: Task label -> cell, in file order D1…Dn (file line 5). Read-only
            mapping; the parser wraps it in ``types.MappingProxyType``.
        charger: Charging station cell (file line 6); start and end of the
            robot's journey.
    """

    rows: int
    cols: int
    obstacles: frozenset[State]
    dirty: Mapping[Label, State]
    charger: State


@dataclass(frozen=True)
class Constraints:
    """Scheduling problem loaded from Schedule.txt (spec §6.2-§6.3).

    Slot exclusivity is implicit and not stored (REQ-CSP-07).

    Attributes:
        variables: Every task label in file order, including tasks without
            constraints (REQ-CSP-03). Carried here because
            select_unassigned_variable receives no ``variables`` argument but
            needs this order for its final tie-break (DEC-05).
        num_slots: Number of available time slots (file line 1).
        precedence: ``(before, after)`` pairs: ``before`` must get a strictly
            earlier slot (REQ-CSP-05).
        latest: Task label -> latest allowed slot (REQ-CSP-06). Read-only
            mapping; the parser wraps it in ``types.MappingProxyType``.
    """

    variables: tuple[Label, ...]
    num_slots: int
    precedence: tuple[tuple[Label, Label], ...]
    latest: Mapping[Label, Slot]


def initial_domains(constraints: Constraints) -> dict[Label, list[Slot]]:
    """Build the initial CSP domains (spec §6.2, REQ-CSP-04).

    Lists in ascending order keep value iteration, and hence the schedule
    found, deterministic (DEC-06). Each variable gets its own list so that
    pruning one domain never affects another.

    Args:
        constraints: Scheduling problem; supplies variables and slot count.

    Returns:
        Mapping from every variable to ``[1, …, num_slots]``.
    """
    return {v: list(range(1, constraints.num_slots + 1)) for v in constraints.variables}


class InputError(ValueError):
    """Invalid Environment.txt or Schedule.txt content (DEC-07).

    One class for both files so callers and tests catch a single type. The
    message always starts with ``"<source>: line <n>:"``.
    """


# One "(row,col)" tuple. "-?" lets a negative number parse so the error can
# say "outside the grid" rather than the vaguer "malformed" (DEC-11).
_CELL_RE = re.compile(r"\(\s*(-?\d+)\s*,\s*(-?\d+)\s*\)")


def _numbered_lines(text: str) -> list[tuple[int, str]]:
    """Split text into ``(line_number, stripped_line)`` pairs.

    Trailing blank lines are dropped; ``splitlines`` also handles Windows
    (CRLF) line endings (DEC-08).

    Args:
        text: Whole file content.

    Returns:
        1-based line numbers paired with whitespace-stripped lines.
    """
    lines = [(i, line.strip()) for i, line in enumerate(text.splitlines(), start=1)]
    while lines and not lines[-1][1]:
        lines.pop()
    return lines


class _LineReader:
    """Hands out lines one at a time and builds located error messages."""

    def __init__(self, text: str, source: str) -> None:
        """Prepare to read ``text``.

        Args:
            text: Whole file content.
            source: Name used in error messages (usually the file name).
        """
        self.lines = _numbered_lines(text)
        self.pos = 0
        self.source = source

    def error(self, lineno: int, problem: str) -> InputError:
        """Build an InputError in the DEC-07 format.

        Args:
            lineno: 1-based line number the problem is on.
            problem: What is wrong, naming the bad value.

        Returns:
            The exception (the caller raises it).
        """
        return InputError(f"{self.source}: line {lineno}: {problem}")

    def next(self, what: str) -> tuple[int, str]:
        """Return the next line; blank lines between items are errors (DEC-08).

        Args:
            what: Description of the expected item, for the error message.

        Returns:
            ``(line_number, stripped_line)``.

        Raises:
            InputError: If the file ended or the line is blank.
        """
        if self.pos >= len(self.lines):
            last = self.lines[-1][0] if self.lines else 0
            raise self.error(last + 1, f"missing {what} (file ended)")
        lineno, line = self.lines[self.pos]
        self.pos += 1
        if not line:
            raise self.error(lineno, f"expected {what}, found a blank line")
        return lineno, line

    def skip_empty_line(self) -> None:
        """Consume one blank line if it is next (empty tuple line, DEC-09)."""
        if self.pos < len(self.lines) and not self.lines[self.pos][1]:
            self.pos += 1

    def expect_end(self) -> None:
        """Raise if any line is left over (DEC-11, DEC-12).

        Raises:
            InputError: On the first unexpected line.
        """
        if self.pos < len(self.lines):
            lineno, line = self.lines[self.pos]
            raise self.error(lineno, f"unexpected extra line {line!r}")


def _parse_count(reader: _LineReader, what: str) -> int:
    """Read a line holding one non-negative integer.

    Args:
        reader: Line source.
        what: Name of the count, for error messages.

    Returns:
        The count.

    Raises:
        InputError: If the line is not a non-negative integer (DEC-11).
    """
    lineno, line = reader.next(what)
    if not re.fullmatch(r"\d+", line):
        raise reader.error(
            lineno, f"{what} must be a non-negative integer, got {line!r}"
        )
    return int(line)


def _parse_cells(line: str) -> list[State] | None:
    """Parse a line of ``(row,col)`` tuples separated by commas/spaces (DEC-10).

    Args:
        line: Stripped line, possibly empty.

    Returns:
        Cells in line order, or None if anything besides tuples, commas and
        spaces is on the line.
    """
    # Whatever is left after removing the tuples must only be separators.
    leftover = _CELL_RE.sub(" ", line)
    if leftover.replace(",", " ").strip():
        return None
    return [(int(r), int(c)) for r, c in _CELL_RE.findall(line)]


def parse_environment(text: str, source: str = "Environment.txt") -> Grid:
    """Parse Environment.txt content into a Grid (spec §5.2, REQ-ENV-02).

    Format, one item or group per line: ``rows,cols``; obstacle count;
    obstacle tuples; dirty count; dirty tuples; charger tuple. Validation rules
    are DEC-07 to DEC-11.

    Args:
        text: Whole file content.
        source: Name used in error messages.

    Returns:
        The validated Grid; dirty labels D1…Dn in file order (REQ-ENV-04).

    Raises:
        InputError: On the first format or validation problem found.
    """
    reader = _LineReader(text, source)

    lineno, line = reader.next("grid dimensions 'rows,cols'")
    match = re.fullmatch(r"(\d+)\s*,\s*(\d+)", line)
    if not match:
        raise reader.error(lineno, f"grid dimensions must be 'rows,cols', got {line!r}")
    rows, cols = int(match[1]), int(match[2])
    if rows <= 0 or cols <= 0:
        raise reader.error(
            lineno, f"grid dimensions must be positive, got {rows}x{cols}"
        )

    # cell -> (role, line it came from); used for the overlap check (§4).
    seen: dict[State, tuple[str, int]] = {}

    def read_cells(what: str, count: int) -> list[State]:
        """Read one tuple line of exactly ``count`` distinct in-bounds cells."""
        if count == 0:
            reader.skip_empty_line()  # DEC-09: empty line or none at all
            return []
        lineno, line = reader.next(f"{what} coordinates")
        cells = _parse_cells(line)
        if cells is None:
            raise reader.error(lineno, f"malformed {what} coordinates {line!r}")
        if len(cells) != count:
            raise reader.error(
                lineno, f"expected {count} {what} cells, found {len(cells)}"
            )
        for cell in cells:
            check_cell(cell, what, lineno)
        return cells

    def check_cell(cell: State, what: str, lineno: int) -> None:
        """Reject out-of-bounds, duplicate and overlapping cells (DEC-11)."""
        r, c = cell
        if not (0 <= r < rows and 0 <= c < cols):
            raise reader.error(lineno, f"{what} cell {cell} outside {rows}x{cols} grid")
        if cell in seen:
            other, other_line = seen[cell]
            if other == what:
                raise reader.error(lineno, f"duplicate {what} cell {cell}")
            raise reader.error(
                lineno, f"{what} cell {cell} overlaps {other} cell (line {other_line})"
            )
        seen[cell] = (what, lineno)

    obstacles = read_cells("obstacle", _parse_count(reader, "obstacle count"))
    dirty_cells = read_cells("dirty", _parse_count(reader, "dirty count"))

    lineno, line = reader.next("charging station coordinate")
    charger_cells = _parse_cells(line)
    if charger_cells is None or len(charger_cells) != 1:
        raise reader.error(
            lineno, f"charging station must be one (row,col) tuple, got {line!r}"
        )
    charger = charger_cells[0]
    check_cell(charger, "charging station", lineno)

    reader.expect_end()

    # D1…Dn in file order (REQ-ENV-04); dicts keep insertion order.
    dirty = {f"D{i}": cell for i, cell in enumerate(dirty_cells, start=1)}
    return Grid(
        rows=rows,
        cols=cols,
        obstacles=frozenset(obstacles),
        dirty=MappingProxyType(dirty),
        charger=charger,
    )


def load_environment(path: str | Path) -> Grid:
    """Load and validate an Environment.txt file (REQ-ENV-01).

    Args:
        path: File to read; supplied by the caller, never hardcoded.

    Returns:
        The validated Grid.

    Raises:
        InputError: If the content is invalid (message names the file).
        OSError: If the file cannot be read.
    """
    path = Path(path)
    return parse_environment(path.read_text(encoding="utf-8"), source=path.name)


# One "(a,b)" pair per line (DEC-10). Items are captured loosely so that the
# error can name a bad label or slot instead of just "malformed".
_PAIR_RE = re.compile(r"\(\s*([^,()\s]+)\s*,\s*([^,()\s]+)\s*\)")
# Slots written "T3" as in the sample, or plain "3" (DEC-12).
_SLOT_RE = re.compile(r"T?(\d+)")


def parse_schedule(text: str, grid: Grid, source: str = "Schedule.txt") -> Constraints:
    """Parse Schedule.txt content into Constraints (spec §6.3, REQ-CSP-02).

    Format: slot count; precedence count; one ``(before,after)`` pair per line;
    latest-slot count; one ``(task,slot)`` pair per line. Validation rules are
    DEC-07 to DEC-10 and DEC-12. Feasibility is not checked here: an
    unsatisfiable but well-formed file is for the CSP solver to report
    (REQ-CSP-12).

    Args:
        text: Whole file content.
        grid: Environment the tasks refer to; supplies the labels D1…Dn.
        source: Name used in error messages.

    Returns:
        Constraints whose ``variables`` are all dirty labels in file order,
        including tasks no constraint mentions (REQ-CSP-03).

    Raises:
        InputError: On the first format or validation problem found.
    """
    reader = _LineReader(text, source)
    labels = tuple(grid.dirty)

    lineno = reader.lines[0][0] if reader.lines else 1
    num_slots = _parse_count(reader, "number of time slots")
    # Slot exclusivity means each task needs its own slot (spec §6.4, REQ-CSP-08).
    if num_slots < len(labels):
        raise reader.error(
            lineno, f"{num_slots} time slots but {len(labels)} dirty locations"
        )

    def read_pair(what: str) -> tuple[int, str, str]:
        """Read one ``(a,b)`` line; return its line number and both items."""
        lineno, line = reader.next(f"{what} pair")
        match = _PAIR_RE.fullmatch(line)
        if not match:
            # A bare number here usually means the count above is too high.
            hint = " (count too high?)" if line.isdigit() else ""
            raise reader.error(lineno, f"malformed {what} pair {line!r}{hint}")
        return lineno, match[1], match[2]

    def check_label(label: str, lineno: int) -> None:
        """Labels must name a dirty cell of the grid (spec §6.4, REQ-CSP-08)."""
        if label not in grid.dirty:
            raise reader.error(
                lineno,
                f"unknown task {label!r} (grid has {', '.join(labels) or 'none'})",
            )

    precedence: list[tuple[Label, Label]] = []
    for _ in range(_parse_count(reader, "number of precedence constraints")):
        lineno, before, after = read_pair("precedence")
        check_label(before, lineno)
        check_label(after, lineno)
        # A repeated pair adds nothing, so it is kept once (DEC-12). (X,X) is
        # kept: it is infeasible, not malformed, so the solver reports it.
        if (before, after) not in precedence:
            precedence.append((before, after))

    latest: dict[Label, Slot] = {}
    for _ in range(_parse_count(reader, "number of latest-slot constraints")):
        lineno, task, slot_text = read_pair("latest-slot")
        check_label(task, lineno)
        match = _SLOT_RE.fullmatch(slot_text)
        if not match:
            raise reader.error(
                lineno, f"slot must look like T3 or 3, got {slot_text!r}"
            )
        slot = int(match[1])
        if not 1 <= slot <= num_slots:
            raise reader.error(lineno, f"slot {slot_text} outside T1..T{num_slots}")
        # Two limits for one task could disagree; latest holds one (DEC-12).
        if task in latest:
            raise reader.error(lineno, f"second latest-slot constraint for {task}")
        latest[task] = slot

    reader.expect_end()

    return Constraints(
        variables=labels,
        num_slots=num_slots,
        precedence=tuple(precedence),
        latest=MappingProxyType(latest),
    )


def load_schedule(path: str | Path, grid: Grid) -> Constraints:
    """Load and validate a Schedule.txt file (REQ-CSP-01).

    Args:
        path: File to read; supplied by the caller, never hardcoded.
        grid: Environment the tasks refer to.

    Returns:
        The validated Constraints.

    Raises:
        InputError: If the content is invalid (message names the file).
        OSError: If the file cannot be read.
    """
    path = Path(path)
    return parse_schedule(path.read_text(encoding="utf-8"), grid, source=path.name)

### Display helpers

Text output used throughout the notebook (spec §6.4, §7.2 output requirements): the grid
drawing, the schedule table with a ✓/✗ check of every constraint (computed independently of
the solver, so it is a real proof), and the integration report. They only format results;
they make no decisions.

In [ ]:
"""Display of grid, paths and schedule (spec §7.2 Output Requirements). Owner: Qazi.

Spec: "The integrated program must display:
- The cleaning schedule generated by the CSP solver.
- The sequence of dirty locations visited by the robot.
- The path computed for each journey between locations.
- The final grid showing the completed cleaning task.
- The total number of movements and total navigation cost for the complete
  journey.
- The final status indicating whether all dirty locations were cleaned and
  the robot returned to the charging station."

Spec §6.4: "Display the generated schedule and show that each constraint is
satisfied" (REQ-CSP-19).

All functions return strings; the caller prints them, so tests can compare
text and the notebook output is plain and screenshot-friendly.
"""

from collections.abc import Mapping
from dataclasses import dataclass

PASS, FAIL = "✓", "✗"

# Grid symbols (display choice). Cleaned cells use lower case "d" so the task
# number stays visible after cleaning.
LEGEND = "Legend: C charger, ## obstacle, Dk dirty, dk cleaned, * path, . free"


@dataclass(frozen=True)
class Check:
    """One constraint checked against a complete assignment (REQ-CSP-19).

    Attributes:
        kind: ``"precedence"``, ``"latest"``, ``"exclusivity"`` or
            ``"assigned"``.
        text: Human-readable evidence, e.g. ``"D1 (T1) before D3 (T3)"``.
        ok: Whether the constraint holds.
    """

    kind: str
    text: str
    ok: bool


def check_schedule(
    assignment: Mapping[Label, Slot], constraints: Constraints
) -> list[Check]:
    """Check every constraint of the problem against an assignment.

    Independent of the solver's own checks, so it is an actual proof that
    the schedule it returned is valid, not a repeat of the same code.

    Args:
        assignment: Label -> slot (complete or not).
        constraints: Scheduling problem.

    Returns:
        One ``Check`` per task-assigned test, per precedence pair, per
        latest-slot limit, plus one for slot exclusivity.
    """
    checks = []
    for v in constraints.variables:
        slot = assignment.get(v)
        ok = slot is not None and 1 <= slot <= constraints.num_slots
        shown = "unassigned" if slot is None else f"T{slot}"
        checks.append(Check("assigned", f"{v} assigned a slot: {shown}", ok))

    for before, after in constraints.precedence:
        b, a = assignment.get(before), assignment.get(after)
        ok = b is not None and a is not None and b < a
        checks.append(
            Check(
                "precedence",
                f"{before} ({_slot(b)}) before {after} ({_slot(a)})",
                ok,
            )
        )

    for task, limit in constraints.latest.items():
        s = assignment.get(task)
        ok = s is not None and s <= limit
        checks.append(Check("latest", f"{task} ≤ T{limit}: {_slot(s)}", ok))

    # Exclusivity: no slot used by two tasks (REQ-CSP-07).
    by_slot: dict[Slot, list[Label]] = {}
    for v, s in assignment.items():
        by_slot.setdefault(s, []).append(v)
    clashes = [
        f"T{s}: {', '.join(vs)}" for s, vs in sorted(by_slot.items()) if len(vs) > 1
    ]
    text = "slot exclusivity: " + (
        "; ".join(clashes) if clashes else "every slot ≤ 1 task"
    )
    checks.append(Check("exclusivity", text, not clashes))
    return checks


def _slot(slot: Slot | None) -> str:
    """Format a slot as ``T<n>``, or ``"-"`` when missing."""
    return "-" if slot is None else f"T{slot}"


def format_schedule(assignment: Mapping[Label, Slot], num_slots: int) -> str:
    """Slot table T1…Tn -> task; empty slots shown as ``-`` (REQ-CSP-19).

    Args:
        assignment: Label -> slot.
        num_slots: Number of available slots.

    Returns:
        Multi-line table text.
    """
    task_in = {s: v for v, s in assignment.items()}
    lines = ["Slot | Task", "-----+-----"]
    for s in range(1, num_slots + 1):
        lines.append(f"{'T' + str(s):<4} | {task_in.get(s, '-')}")
    return "\n".join(lines)


def format_schedule_proof(
    assignment: Mapping[Label, Slot] | None, constraints: Constraints
) -> str:
    """Schedule table plus each constraint with ✓/✗ (spec §6.4, REQ-CSP-19).

    Args:
        assignment: Solver result, or None if no feasible schedule exists.
        constraints: Scheduling problem.

    Returns:
        Text to print.
    """
    if assignment is None:
        return "No feasible schedule: the CSP solver reports failure."
    checks = check_schedule(assignment, constraints)
    lines = [format_schedule(assignment, constraints.num_slots), "", "Constraints:"]
    lines += [f"  {c.text} {PASS if c.ok else FAIL}" for c in checks]
    verdict = "all constraints satisfied" if all(c.ok for c in checks) else "VIOLATED"
    lines.append(f"Result: {verdict}")
    return "\n".join(lines)


def format_grid(
    grid: Grid,
    path: list[State] | None = None,
    cleaned: list[Label] | tuple[Label, ...] = (),
) -> str:
    """Draw the grid as text with row/column numbers (REQ-INT-08).

    Args:
        grid: Environment.
        path: States to mark with ``*`` (stops keep their own symbol).
        cleaned: Labels already cleaned, drawn as ``dk``.

    Returns:
        Multi-line grid text, row 0 at the top.
    """
    cells: dict[State, str] = {}
    for state in path or []:
        cells[state] = "*"
    for cell in grid.obstacles:
        cells[cell] = "##"
    for label, cell in grid.dirty.items():
        cells[cell] = label.lower() if label in cleaned else label
    cells[grid.charger] = "C"

    width = 3
    header = " " * width + "".join(f"{c:>{width}}" for c in range(grid.cols))
    lines = [header]
    for r in range(grid.rows):
        row = "".join(f"{cells.get((r, c), '.'):>{width}}" for c in range(grid.cols))
        lines.append(f"{r:>{width}}{row}")
    return "\n".join(lines)


def _stop(label: str) -> str:
    """Name a stop for display: the charger is shown as ``C``."""
    return "C" if label == CHARGER else label


# "IntegrationResult" is quoted so that this module's code can be placed before
# integrate.py's in the notebook (#27); it is only needed when called.
def format_legs(result: "IntegrationResult") -> str:
    """One block per search call: path, cost, nodes expanded (REQ-INT-08).

    Args:
        result: Integration result.

    Returns:
        Text to print.
    """
    lines = []
    for i, leg in enumerate(result.legs, start=1):
        head = (
            f"Leg {i}: {_stop(leg.source)} {leg.start} -> "
            f"{_stop(leg.target)} {leg.goal}"
        )
        if leg.path is None:
            lines.append(f"{head}: NO PATH (nodes expanded {leg.nodes_expanded})")
        else:
            steps = " -> ".join(str(s) for s in leg.path)
            lines.append(
                f"{head}: cost {leg.cost}, nodes expanded {leg.nodes_expanded}"
            )
            lines.append(f"  {steps}")
    return "\n".join(lines)


def format_result(result: "IntegrationResult", grid: Grid) -> str:
    """Full integration report, in spec §7.2 Output Requirements order.

    Args:
        result: Integration result from ``plan_and_execute``/``run_integration``.
        grid: Environment it ran on.

    Returns:
        Text to print.
    """
    lines = [f"=== Integration ({result.algorithm}) ==="]
    if result.assignment is not None:
        order = [f"T{result.assignment[x]}:{x}" for x in result.schedule]
    else:
        order = list(result.schedule)
    lines.append("Schedule: " + (", ".join(order) if order else "none"))
    visited = ["C"] + result.cleaned + (["C"] if result.success else [])
    lines.append("Visit sequence: " + " -> ".join(visited))
    lines += ["", "Paths:", format_legs(result) or "  (no search run)", ""]
    lines += ["Final grid:", format_grid(grid, result.complete_path, result.cleaned)]
    lines.append(LEGEND)
    moves = len(result.complete_path) - 1
    lines.append(f"Total moves: {moves}, total cost: {result.total_cost}")
    if result.success:
        status = "SUCCESS: all dirty locations cleaned, robot back at charger"
    else:
        left = [x for x in grid.dirty if x not in result.cleaned]
        status = (
            f"FAILURE: {result.reason}; cleaned {result.cleaned or 'none'}, "
            f"not cleaned {left}, robot did not return to charger"
        )
    lines.append("Status: " + status)
    return "\n".join(lines)

### Our environment

In [ ]:
grid = load_environment(find_file(ENV_FILE))
constraints = load_schedule(find_file(SCHEDULE_FILE), grid)
print(
    f"Grid {grid.rows}x{grid.cols}, {len(grid.obstacles)} obstacles, "
    f"charger {grid.charger}"
)
print("Dirty cells:", dict(grid.dirty))
print(
    f"Slots: {constraints.num_slots}, precedence: {list(constraints.precedence)}, "
    f"latest: {dict(constraints.latest)}"
)
print()
print(format_grid(grid))
print(LEGEND)

## 2. Graph-based search

- **State:** `(row, col)`; **actions:** UP, DOWN, LEFT, RIGHT (in that order, which fixes
  tie-breaking); a move is valid if it stays inside the grid and does not enter an obstacle;
  **step cost:** 1; **goal test:** `state == goal_state` (spec §5.3).
- **Algorithms (DEC-01):** Uniform-Cost Search (uninformed) and A\* (informed). Both use the
  same best-first graph search with a priority queue; UCS orders by `g(n)`, A\* by
  `g(n) + h(n)`. The goal is tested when a node is taken off the queue, which is what makes
  both optimal. A counter breaks equal priorities first-in-first-out, so results are
  deterministic.
- **Nodes expanded (DEC-14):** a node counts when its successors are generated; the goal node
  is not counted, so start == goal gives `([start], 0, 0)`.
- **No path:** `search` returns `(None, None, nodes_expanded)`; an unknown algorithm name
  raises `ValueError` so a typo is never mistaken for "no path".
- `path_cost([])` returns 0 (DEC-04).

In [ ]:
"""Graph-based search for robot navigation (spec §5). Owner: Mujtaba.

Spec §5.3: "State: Represent the robot's position as (row,column)." /
"Actions: UP, DOWN, LEFT, and RIGHT. An action is valid only if the resulting
cell is within the grid boundaries and is not an obstacle." / "Step cost: Each
valid movement has a cost of 1."

Frozen interface: see docs/plan.md §4 and AGENTS.md.

UCS and A* share one best-first graph search (DEC-01); A* only adds the
heuristic to the priority (REQ-SRCH-01).
"""

import heapq
from collections.abc import Sequence
from itertools import count

# Spec §5.3: "Each valid movement has a cost of 1."
STEP_COST = 1


def get_successors(state: State, grid: Grid) -> list[tuple[State, str, int]]:
    """Find all valid successor states (spec §5.5).

    Spec: "Find all valid successor states without including cells outside
    the grid or obstacle cells."

    Args:
        state: Current robot position.
        grid: Environment representation.

    Returns:
        A list of valid successors, each represented as
        (next_state, action, step_cost).
    """
    row, col = state
    successors = []
    # ACTIONS order fixes successor order and hence tie-breaks (REQ-SRCH-02).
    for action, (dr, dc) in ACTIONS:
        nxt = (row + dr, col + dc)
        in_bounds = 0 <= nxt[0] < grid.rows and 0 <= nxt[1] < grid.cols
        if in_bounds and nxt not in grid.obstacles:
            successors.append((nxt, action, STEP_COST))
    return successors


def goal_test(state: State, goal_state: State) -> bool:
    """Check whether the goal has been reached (spec §5.5).

    Args:
        state: Current robot position.
        goal_state: Target position.

    Returns:
        True if state equals goal_state; otherwise, False.
    """
    return state == goal_state


def path_cost(path: Sequence[State]) -> int:
    """Calculate the total cost of a path (spec §5.5).

    Spec: "Since each movement costs 1, the path cost equals the number of
    movements, or the number of states in the path minus one. An empty path
    and a path containing only the initial state must be handled
    appropriately."

    Args:
        path: Sequence of states from the start to the goal.

    Returns:
        Total path cost.
    """
    # DEC-04: [] -> 0 (team choice); a single-state path has no moves.
    if not path:
        return 0
    return (len(path) - 1) * STEP_COST


def heuristic(state: State, goal_state: State) -> int:
    """Estimate the cost from the current state to the goal (spec §5.5).

    Spec: "For four-directional movement, students may use Manhattan distance:
    h(n) = |row_n - row_goal| + |col_n - col_goal|. Students must explain why
    their selected heuristic is appropriate for the environment and search
    algorithm."

    Args:
        state: Current robot position.
        goal_state: Target position.

    Returns:
        An estimate of the cost from the current state to the goal.
    """
    # Manhattan distance (DEC-02): admissible and consistent for 4-way unit
    # moves, since each move changes |dr| + |dc| by exactly 1 (REQ-SRCH-08).
    return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])


def search(
    grid: Grid, start: State, goal: State, algorithm: str
) -> tuple[list[State] | None, int | None, int]:
    """Implement the selected search algorithm (spec §5.5).

    Spec §5.4: "The start and goal positions must be provided as inputs rather
    than hardcoded in the search algorithm." / "The program must correctly
    handle cases where the start equals the goal or no path exists."

    Args:
        grid: Environment representation.
        start: Starting position.
        goal: Target position.
        algorithm: Selected graph-search algorithm, one of ``ALGORITHMS``.

    Returns:
        path: Sequence of states from start to goal, or None if no path
            exists.
        cost: Total path cost if a path is found, otherwise None.
        nodes_expanded: Number of nodes expanded during the search.

    Raises:
        ValueError: If ``algorithm`` is not in ``ALGORITHMS``.
    """
    if algorithm not in ALGORITHMS:
        # DEC-06: a typo must not look like "no path exists".
        raise ValueError(
            f"unknown algorithm {algorithm!r}; expected one of {ALGORITHMS}"
        )
    use_h = algorithm == "astar"

    def priority(state: State, g: int) -> int:
        return g + heuristic(state, goal) if use_h else g

    # The counter breaks equal priorities first-in-first-out, so results are
    # deterministic and follow ACTIONS order (REQ-SRCH-02).
    tie = count()
    frontier = [(priority(start, 0), next(tie), start)]
    best_g = {start: 0}
    parent: dict[State, State | None] = {start: None}
    closed: set[State] = set()
    nodes_expanded = 0

    while frontier:
        _, _, state = heapq.heappop(frontier)
        if state in closed:
            continue  # stale duplicate left behind by a cheaper re-push
        # Goal test on pop, not on generation: required for UCS/A* optimality.
        # The goal node is not counted as expanded (its successors are never
        # generated), so start == goal gives 0 expansions (REQ-SRCH-13).
        if goal_test(state, goal):
            path = _reconstruct(parent, state)
            return path, path_cost(path), nodes_expanded
        closed.add(state)
        nodes_expanded += 1
        for nxt, _action, step in get_successors(state, grid):
            g = best_g[state] + step
            if nxt not in closed and g < best_g.get(nxt, g + 1):
                best_g[nxt] = g
                parent[nxt] = state
                heapq.heappush(frontier, (priority(nxt, g), next(tie), nxt))

    # Frontier exhausted: goal unreachable (REQ-SRCH-14).
    return None, None, nodes_expanded


def _reconstruct(parent: dict[State, State | None], state: State) -> list[State]:
    """Follow parent links back to the start.

    Args:
        parent: Child -> parent map built by search; the start maps to None.
        state: Goal state reached.

    Returns:
        States from start to ``state`` inclusive.
    """
    path = []
    node: State | None = state
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path

In [ ]:
# Each algorithm from the charger to the first dirty cell, plus the edge cases.
first = next(iter(grid.dirty))
for algorithm in ALGORITHMS:
    path, cost, nodes = search(grid, grid.charger, grid.dirty[first], algorithm)
    print(f"{algorithm}: C -> {first}: cost {cost}, nodes expanded {nodes}")
    print("  path:", path)
print("start == goal:", search(grid, grid.charger, grid.charger, "astar"))
print("path_cost([]) =", path_cost([]))

### Why Manhattan distance is the right heuristic for A*

The spec (§5.5) lets us use Manhattan distance, `h(n) = |row_n − row_goal| + |col_n − col_goal|`,
and asks us to explain why the heuristic suits the environment and the algorithm (DEC-02).

**Admissible (never overestimates).** The robot moves only UP, DOWN, LEFT or RIGHT, and every
move costs 1 (§5.3). One move changes either the row or the column by exactly 1, so it can reduce
the Manhattan distance by at most 1. Reaching the goal therefore takes at least `h(n)` moves, and
obstacles can only make the real path longer, never shorter. So `h(n) ≤ h*(n)` for every cell.

**Consistent (monotone).** For any move from `n` to a neighbour `n'` with cost `c(n, n') = 1`,
the Manhattan distance changes by exactly ±1, so `h(n) ≤ c(n, n') + h(n')`. Also `h(goal) = 0`.
Consistency implies admissibility, and it means `f = g + h` never decreases along a path.

**Why this matters for A\*.** Our A\* is a graph search (it keeps an explored set). With a
consistent heuristic, the first time A\* expands a cell it has already found the cheapest path to
it, so it never has to reopen a cell and the path it returns is optimal — the same cost UCS finds.

**Why not something else.** Euclidean distance is also admissible but is never larger than
Manhattan on this grid, so it is less informed and A\* would expand more nodes. `h = 0` turns A\*
into UCS. A heuristic that ignores the four-direction rule (e.g. Chebyshev, for diagonal moves)
would be weaker still. Manhattan is the largest of these that is still admissible, so it gives
the fewest expansions without losing optimality.

### Comparison code

Uses only the `search` function above, so the comparison measures the same code that the
integration runs.

In [ ]:
"""UCS vs A* comparison (spec §5.4, §8.2; REQ-CMP-01, REQ-CMP-02). Owner: Mujtaba.

Spec §5.4: "Compare the implemented algorithms in terms of path cost, nodes
expanded, and their behavior and performance." Spec §8.2: "Present tabular
and/or graphical comparisons..."

Only the frozen ``search`` interface is used, so the comparison measures the
same code the notebook and integration run.
"""

import time
from collections.abc import Iterable, Sequence
from dataclasses import dataclass


@dataclass(frozen=True)
class Run:
    """One search run for the comparison table (REQ-CMP-01)."""

    label: str
    algorithm: str
    start: State
    goal: State
    path: list[State] | None
    cost: int | None
    nodes_expanded: int
    seconds: float


def run_pair(
    grid: Grid, start: State, goal: State, label: str = "", repeats: int = 1
) -> list[Run]:
    """Run every algorithm on one start/goal pair.

    Args:
        grid: Environment representation.
        start: Starting position.
        goal: Target position.
        label: Name shown in the table.
        repeats: Timing repetitions; the minimum is kept because it is the
            least affected by OS noise.

    Returns:
        One ``Run`` per algorithm, in ``ALGORITHMS`` order.
    """
    runs = []
    for algorithm in ALGORITHMS:
        best = float("inf")
        for _ in range(max(1, repeats)):
            t0 = time.perf_counter()
            path, cost, nodes = search(grid, start, goal, algorithm)
            best = min(best, time.perf_counter() - t0)
        runs.append(Run(label, algorithm, start, goal, path, cost, nodes, best))
    return runs


def compare_pairs(
    grid: Grid, pairs: Iterable[tuple[str, State, State]], repeats: int = 1
) -> list[Run]:
    """Compare all algorithms on several labelled start/goal pairs.

    Args:
        grid: Environment representation.
        pairs: ``(label, start, goal)`` triples.
        repeats: Timing repetitions per run.

    Returns:
        Runs for every pair and algorithm.
    """
    runs: list[Run] = []
    for label, start, goal in pairs:
        runs.extend(run_pair(grid, start, goal, label, repeats))
    return runs


def charger_pairs(grid: Grid) -> list[tuple[str, State, State]]:
    """Start/goal pairs from the environment file: charger -> each dirty cell.

    Taking pairs from ``grid`` keeps coordinates out of the code (AGENTS.md),
    and these are exactly the legs integration plans.

    Args:
        grid: Environment representation.

    Returns:
        ``(label, charger, cell)`` per dirty cell in file order.
    """
    return [(f"C->{lbl}", grid.charger, cell) for lbl, cell in grid.dirty.items()]


def open_grid(size: int) -> Grid:
    """Obstacle-free ``size`` x ``size`` grid for the scaling experiment.

    Args:
        size: Number of rows and columns.

    Returns:
        A grid with no obstacles or dirty cells.
    """
    return Grid(size, size, frozenset(), {}, (0, 0))


SCENARIOS = ("corner", "midline")


def scaling(
    sizes: Sequence[int], repeats: int = 3, scenario: str = "corner"
) -> list[Run]:
    """Runs on open grids of growing size.

    ``corner``: (0,0) -> opposite corner. Every cell inside the bounding box
    has the same f = g + h, so A* has nothing to prune and ties with UCS.
    ``midline``: left-edge midpoint -> right-edge midpoint; only cells on the
    straight line keep f minimal, which shows A*'s best case.

    Args:
        sizes: Grid side lengths to test.
        repeats: Timing repetitions per run.
        scenario: One of ``SCENARIOS``.

    Returns:
        Runs labelled ``"<n>x<n>"``.

    Raises:
        ValueError: If ``scenario`` is unknown.
    """
    if scenario not in SCENARIOS:
        raise ValueError(f"unknown scenario {scenario!r}; expected {SCENARIOS}")
    runs: list[Run] = []
    for n in sizes:
        if scenario == "corner":
            start, goal = (0, 0), (n - 1, n - 1)
        else:
            start, goal = (n // 2, 0), (n // 2, n - 1)
        runs.extend(run_pair(open_grid(n), start, goal, f"{n}x{n}", repeats))
    return runs


def format_table(runs: Sequence[Run]) -> str:
    """Render runs as a Markdown table (spec §8.2 tabular comparison).

    Args:
        runs: Runs to show.

    Returns:
        Markdown table text.
    """
    lines = [
        "| Case | Algorithm | Start | Goal | Path cost | Nodes expanded | Time (ms) |",
        "|---|---|---|---|---|---|---|",
    ]
    for r in runs:
        cost = "no path" if r.cost is None else str(r.cost)
        lines.append(
            f"| {r.label} | {r.algorithm} | {r.start} | {r.goal} | {cost} "
            f"| {r.nodes_expanded} | {r.seconds * 1000:.3f} |"
        )
    return "\n".join(lines)

### UCS vs A\* on our environment

Each pair runs both algorithms from the charging station to each dirty cell. Path costs
must be equal (both are optimal); the difference is in nodes expanded. The scaling runs use
open n×n grids: corner-to-corner (A\* has nothing to prune, every cell in the box has the same
`f`) and edge-midpoint to edge-midpoint (A\*'s best case).

In [ ]:
import matplotlib.pyplot as plt

pair_runs = compare_pairs(grid, charger_pairs(grid), repeats=3)
print(format_table(pair_runs))

sizes = [5, 10, 20, 40]
scale = {sc: scaling(sizes, repeats=1, scenario=sc) for sc in SCENARIOS}
for sc, runs in scale.items():
    print(f"\nScaling, open grids, {sc}:")
    print(format_table(runs))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = list(dict.fromkeys(r.label for r in pair_runs))
width = 0.8 / len(ALGORITHMS)
for i, alg in enumerate(ALGORITHMS):
    nodes = [r.nodes_expanded for r in pair_runs if r.algorithm == alg]
    axes[0].bar([x + i * width for x in range(len(labels))], nodes, width, label=alg)
    for sc, runs in scale.items():
        ys = [r.nodes_expanded for r in runs if r.algorithm == alg]
        axes[1].plot(sizes, ys, marker="o", label=f"{alg} ({sc})")
axes[0].set_xticks([x + width / 2 for x in range(len(labels))], labels)
axes[0].set_title("Nodes expanded per start/goal pair")
axes[1].set_title("Scaling on open n x n grids")
axes[1].set_xlabel("grid side n")
for ax in axes:
    ax.set_ylabel("nodes expanded")
    ax.legend()
fig.tight_layout()
plt.show()

## 3. CSP scheduling

- **Variables:** every dirty cell `D1…Dn` (including those no constraint mentions).
- **Domains:** slots `{1…num_slots}`, ascending.
- **Constraints:** precedence `(A,B)` ⇒ slot(A) < slot(B); latest-slot `(A,Tk)` ⇒ slot(A) ≤ k;
  slot exclusivity (implicit) ⇒ no two tasks share a slot.

**Solver** (spec §6.5): backtracking with
- **MRV**, ties broken by the **degree heuristic** — number of distinct other unassigned
  tasks linked by precedence (DEC-05; exclusivity links every pair so it is not counted),
  then file order;
- **LCV** value ordering (fewest values ruled out for the other tasks first);
- **forward checking** after every assignment; a branch is rejected as soon as a domain is
  empty. Forward checking works on a copy of the domains, so backtracking restores the
  removed values automatically.

Values above a task's latest slot are removed before the search starts.

Below the schedule, every constraint is re-checked independently of the solver and printed
with ✓ / ✗ and the slots involved (spec §6.4).

In [ ]:
"""CSP for cleaning-task scheduling (spec §6). Owner: Huzaifa.

Spec §6.2: "Variables: Each dirty location represents a CSP variable." /
"Domains: Each variable initially has the set of available time slots,
{T1,T2,...,Tn}, as its domain." / Precedence, latest-slot and slot-exclusivity
constraints.

Frozen interface: see docs/plan.md §4 and AGENTS.md.

Types (DEC-06): variables are task labels ``"D1"…"Dn"``; slots are ints
``1…n`` (shown as ``T<n>`` only in output); ``domains`` is
``dict[label, list[int]]`` in ascending order; ``assignment`` is
``dict[label, int]``; ``constraints`` is ``env.Constraints``.
"""


Domains = dict[Label, list[Slot]]
Assignment = dict[Label, Slot]


def _pair_ok(
    x: Label, x_slot: Slot, y: Label, y_slot: Slot, constraints: Constraints
) -> bool:
    """Check the binary constraints between two different tasks.

    Slot exclusivity (REQ-CSP-07) and precedence in either direction
    (REQ-CSP-05) are the only constraints that involve two tasks; latest-slot
    is unary and checked separately. Shared by ``is_consistent``,
    ``forward_check`` and ``order_domain_values`` so all three agree on what
    "conflict" means.

    Args:
        x: First task.
        x_slot: Slot proposed for ``x``.
        y: Second task (different from ``x``).
        y_slot: Slot proposed for ``y``.
        constraints: Scheduling constraints.

    Returns:
        True if the two placements do not violate any binary constraint.
    """
    if x_slot == y_slot:
        return False
    for before, after in constraints.precedence:
        if (before, after) == (x, y) and not x_slot < y_slot:
            return False
        if (before, after) == (y, x) and not y_slot < x_slot:
            return False
    return True


def _latest_ok(variable: Label, value: Slot, constraints: Constraints) -> bool:
    """Check the latest-slot constraint of one task (REQ-CSP-06).

    Args:
        variable: Task.
        value: Proposed slot.
        constraints: Scheduling constraints.

    Returns:
        True if ``variable`` has no latest-slot limit or ``value`` is within it.
    """
    limit = constraints.latest.get(variable)
    return limit is None or value <= limit


def _degree(variable: Label, assignment: Assignment, constraints: Constraints) -> int:
    """Count distinct other unassigned tasks linked by precedence (DEC-05).

    Slot exclusivity is not counted: it links every pair of tasks, so it adds
    the same amount to every score and could never change the ranking (#40).

    Args:
        variable: Task whose degree is wanted.
        assignment: Current partial assignment.
        constraints: Scheduling constraints.

    Returns:
        Number of distinct neighbours that are still unassigned.
    """
    neighbours = set()
    for before, after in constraints.precedence:
        if before == variable:
            neighbours.add(after)
        elif after == variable:
            neighbours.add(before)
    # A self-precedence (X,X) names X itself, which is not an "other" variable.
    neighbours.discard(variable)
    return sum(1 for n in neighbours if n not in assignment)


def select_unassigned_variable(assignment, domains, constraints):
    """Select an unassigned variable using MRV, ties broken by degree (spec §6.5).

    Spec: "Selects an unassigned variable using the Minimum Remaining Values
    (MRV), also called the most-constrained-variable heuristic. If multiple
    variables are tied, uses the degree heuristic to select the variable
    connected to the largest number of other unassigned variables through
    constraints."

    Order of tie-breaks (REQ-CSP-13, DEC-05): fewest remaining values, then
    highest degree (precedence links to other unassigned tasks only), then
    file order D1…Dn.

    Args:
        assignment: Current variable assignments.
        domains: Current domains of the variables.
        constraints: Scheduling constraints.

    Returns:
        The next unassigned variable to select, or None if every variable is
        already assigned.
    """
    unassigned = [v for v in constraints.variables if v not in assignment]
    if not unassigned:
        return None
    # min() keeps the first of equal keys, and the list is in file order, so
    # the last tie-break (D1…Dn, DEC-05) needs no extra key.
    return min(
        unassigned,
        key=lambda v: (len(domains[v]), -_degree(v, assignment, constraints)),
    )


def order_domain_values(variable, assignment, domains, constraints):
    """Order candidate time slots using LCV (spec §6.5).

    Spec: "Orders the candidate time slots for the selected variable, using
    Least Constraining Value (LCV) heuristic to order values."

    For each slot, counts how many values it would rule out of the domains of
    the other unassigned tasks (same slot, or wrong side of a precedence).
    Fewest ruled out comes first. Equal counts keep the domain's ascending
    order (DEC-06), because ``sorted`` is stable.

    Args:
        variable: Selected variable.
        assignment: Current partial assignment.
        domains: Current domains.
        constraints: Scheduling constraints.

    Returns:
        An ordered list of candidate time slots (every value in
        ``domains[variable]``; consistency is checked by the caller).
    """
    others = [v for v in constraints.variables if v != variable and v not in assignment]

    def ruled_out(value: Slot) -> int:
        return sum(
            1
            for other in others
            for other_value in domains[other]
            if not _pair_ok(variable, value, other, other_value, constraints)
        )

    return sorted(domains[variable], key=ruled_out)


def is_consistent(variable, value, assignment, constraints):
    """Check precedence, latest-slot and slot-exclusivity (spec §6.5).

    Spec: "Checks precedence, latest-slot, and slot exclusivity constraints
    whenever the necessary assignments are available."

    Only constraints whose tasks are all assigned (counting ``variable`` =
    ``value``) are checked; the rest are left for later (REQ-CSP-15).

    Args:
        variable: Variable being assigned.
        value: Proposed time slot.
        assignment: Current partial assignment.
        constraints: Scheduling constraints.

    Returns:
        True if the proposed assignment does not violate any constraint that
        can be evaluated using the current assignment; otherwise, False.
    """
    if not _latest_ok(variable, value, constraints):  # REQ-CSP-06
        return False
    # (X,X) can never hold: no slot comes strictly before itself (DEC-12).
    if (variable, variable) in constraints.precedence:
        return False
    # Exclusivity and precedence against every task already placed
    # (REQ-CSP-05, REQ-CSP-07). Skipping ``variable`` lets callers pass an
    # assignment that already contains it.
    return all(
        _pair_ok(variable, value, other, slot, constraints)
        for other, slot in assignment.items()
        if other != variable
    )


def forward_check(variable, value, assignment, domains, constraints):
    """Prune infeasible values from related unassigned variables (spec §6.5).

    Spec: "Removes time-slot values that are no longer feasible for related
    unassigned variables. If any unassigned variable has an empty domain, the
    current assignment is rejected. Any domain values removed during forward
    checking must be restored when backtracking to a previous assignment."

    The input ``domains`` is never modified: pruning happens on a copy that is
    returned. Backtracking restores the removed values simply by going on with
    the ``domains`` it had before the call (REQ-CSP-17).

    Args:
        variable: Newly assigned variable.
        value: Assigned time slot.
        assignment: Current assignment.
        domains: Current domains.
        constraints: Scheduling constraints.

    Returns:
        Updated domains if the assignment is consistent and no domain becomes
        empty; otherwise, None (REQ-CSP-16).
    """
    new_domains = {v: list(values) for v, values in domains.items()}
    new_domains[variable] = [value]
    for other in constraints.variables:
        if other == variable or other in assignment:
            continue
        new_domains[other] = [
            v
            for v in new_domains[other]
            if _pair_ok(variable, value, other, v, constraints)
        ]
        if not new_domains[other]:
            return None
    return new_domains


def backtracking_search(variables, domains, constraints):
    """Backtracking CSP solver (spec §6.5).

    Spec: "Implement a backtracking CSP solver using constraint checking,
    forward checking, and the required variable-selection heuristic." /
    "It must perform forward checking after assigning a value and backtrack
    when an assignment leads to failure."

    Before searching, values above a task's latest slot are removed (they can
    never be consistent), so MRV sees those tasks as more constrained from the
    start. Each step: pick a variable (MRV/degree), try its values in LCV
    order, keep a value only if it is consistent and forward checking leaves
    no empty domain; otherwise undo and try the next value (REQ-CSP-18).

    Args:
        variables: CSP variables; must be the same tasks as
            ``constraints.variables`` (every dirty cell, REQ-CSP-03).
        domains: Initial domains (not modified).
        constraints: Scheduling constraints.

    Returns:
        A complete assignment (label -> slot) if a feasible schedule exists;
        otherwise, None (REQ-CSP-11, REQ-CSP-12).

    Raises:
        ValueError: If ``variables`` and ``constraints.variables`` differ.
    """
    if set(variables) != set(constraints.variables):
        raise ValueError("variables must match constraints.variables")

    start = {
        v: [s for s in domains[v] if _latest_ok(v, s, constraints)] for v in variables
    }
    if any(not values for values in start.values()):
        return None
    return _backtrack({}, start, constraints)


def _backtrack(
    assignment: Assignment, domains: Domains, constraints: Constraints
) -> Assignment | None:
    """Recursive step of ``backtracking_search``.

    Args:
        assignment: Partial assignment, extended in place and undone on failure.
        domains: Domains after forward checking for ``assignment``.
        constraints: Scheduling constraints.

    Returns:
        A complete assignment, or None if this branch has no solution.
    """
    variable = select_unassigned_variable(assignment, domains, constraints)
    if variable is None:
        return dict(assignment)
    for value in order_domain_values(variable, assignment, domains, constraints):
        if not is_consistent(variable, value, assignment, constraints):
            continue
        assignment[variable] = value
        pruned = forward_check(variable, value, assignment, domains, constraints)
        if pruned is not None:
            result = _backtrack(assignment, pruned, constraints)
            if result is not None:
                return result
        # Backtrack: drop the value. ``domains`` was never changed, so the
        # values forward checking removed are back for the next try.
        del assignment[variable]
    return None

In [ ]:
assignment = backtracking_search(
    list(constraints.variables), initial_domains(constraints), constraints
)
print(format_schedule_proof(assignment, constraints))

## 4. Integration

`plan_and_execute` runs the CSP solver, sorts the tasks by slot to get the cleaning order,
then calls `execute_schedule`'s procedure (spec §7.2): start at the charger, `search` from the
current position to each next dirty cell, append each path without repeating the shared
endpoint, mark the cell cleaned, and finally `search` back to the charger.

- If the CSP finds no schedule, no search is run and failure is reported.
- If any leg has no path, the run stops and reports failure; it never claims success. The
  path walked so far and its cost are kept so the output shows where the robot got stuck
  (DEC-15).

Output (spec §7.2): schedule, visit sequence, each leg's path with cost and nodes expanded,
final grid (`C` charger, `##` obstacle, `Dk` dirty, `dk` cleaned, `*` path), total moves and
cost, and final status. It is shown below for both algorithms on our files.

In [ ]:
"""Integration of CSP scheduling and graph-based search (spec §7). Owner: Qazi.

Frozen interface: see docs/plan.md §4 and AGENTS.md.

``execute_schedule`` is the frozen function. ``run_integration`` does the same
work but also keeps per-leg details (path, cost, nodes expanded) and a failure
reason, which the display (#24) needs; the frozen return value has no room for
them. ``plan_and_execute`` runs the whole pipeline: CSP first, then search.
"""

from collections.abc import Mapping, Sequence
from dataclasses import dataclass, replace

# Name used for the charging station in leg descriptions and messages.
CHARGER = "C"


@dataclass(frozen=True)
class Leg:
    """One search call between two consecutive stops (spec §7.2 step 3/6).

    Attributes:
        source: Label of the start stop (``"C"`` for the charger).
        target: Label of the goal stop.
        start: Start cell.
        goal: Goal cell.
        path: States from start to goal, or None if no path exists.
        cost: Path cost, or None if no path exists.
        nodes_expanded: Nodes expanded by this search call.
    """

    source: str
    target: str
    start: State
    goal: State
    path: list[State] | None
    cost: int | None
    nodes_expanded: int


@dataclass(frozen=True)
class IntegrationResult:
    """Everything the integrated run produced (spec §7.2 step 8, REQ-INT-08).

    Attributes:
        algorithm: Search algorithm used.
        schedule: Labels in slot order (empty if the CSP failed).
        assignment: CSP assignment label -> slot, or None if the CSP failed or
            the schedule was given directly.
        legs: Search calls made, in order; the last one failed if
            ``success`` is False because of a missing path.
        cleaned: Labels cleaned, in visit order.
        complete_path: Joined path; on failure, the part walked so far
            (DEC-15).
        total_cost: Cost of ``complete_path``.
        success: True only if every task was cleaned and the robot is back at
            the charger (REQ-INT-07).
        reason: Empty on success; otherwise why the run failed.
    """

    algorithm: str
    schedule: list[Label]
    assignment: dict[Label, Slot] | None
    legs: list[Leg]
    cleaned: list[Label]
    complete_path: list[State]
    total_cost: int
    success: bool
    reason: str


def schedule_from_assignment(assignment: Mapping[Label, Slot]) -> list[Label]:
    """Order tasks by their assigned slot (spec §7.1 step 2, REQ-INT-01).

    Slots are ints, so sorting by value is numeric: T10 comes after T9
    (DEC-06).

    Args:
        assignment: CSP assignment label -> slot.

    Returns:
        Labels in slot order.
    """
    return sorted(assignment, key=lambda label: assignment[label])


def run_integration(
    grid: Grid,
    charging_station: State,
    schedule: Sequence[Label],
    algorithm: str,
) -> IntegrationResult:
    """Execute a schedule and keep per-leg details (spec §7.2 procedure).

    Args:
        grid: Environment loaded from Environment.txt.
        charging_station: Start and end cell of the journey.
        schedule: Task labels in slot order; converted to cells via
            ``grid.dirty``.
        algorithm: Search algorithm name, one of ``ALGORITHMS``.

    Returns:
        The full result; see ``IntegrationResult``.

    Raises:
        KeyError: If a label is not a dirty cell of ``grid``.
        ValueError: If ``algorithm`` is unknown (raised by ``search``).
    """
    # Stops: each scheduled task, then back to the charger (steps 2 and 6).
    stops = [(label, grid.dirty[label]) for label in schedule]
    stops.append((CHARGER, charging_station))

    current_label, current = CHARGER, charging_station  # step 1
    complete_path = [charging_station]
    legs: list[Leg] = []
    cleaned: list[Label] = []

    for label, cell in stops:
        path, cost, nodes = search(grid, current, cell, algorithm)  # step 3
        legs.append(Leg(current_label, label, current, cell, path, cost, nodes))
        if path is None:
            # Step 9: stop and never claim success. The path walked so far is
            # kept so the display can show where the robot got stuck (DEC-15).
            reason = (
                f"no path from {current_label} {current} to {label} {cell}"
                f" ({algorithm})"
            )
            return IntegrationResult(
                algorithm,
                list(schedule),
                None,
                legs,
                cleaned,
                complete_path,
                path_cost(complete_path),
                False,
                reason,
            )
        # Step 4: path[0] equals the last state already in complete_path.
        complete_path.extend(path[1:])
        if label != CHARGER:
            cleaned.append(label)  # step 5: reached, so cleaned (REQ-INT-03)
        current_label, current = label, cell

    return IntegrationResult(
        algorithm,
        list(schedule),
        None,
        legs,
        cleaned,
        complete_path,
        path_cost(complete_path),  # step 7
        True,
        "",
    )


def execute_schedule(grid, charging_station, schedule, algorithm):
    """Execute the CSP-generated schedule using graph-based search (spec §7.2).

    Spec procedure (abridged): start at the charging station; for each
    cleaning location in schedule order "call search using the robot's current
    position as the start state and the next cleaning location as the goal
    state"; "Append each returned path to complete_path, avoiding duplication
    of shared endpoints"; finally search back to the charging station. "If a
    required path cannot be found, the program must report failure rather
    than claim that the cleaning task was completed successfully."

    Args:
        grid: Environment loaded from Environment.txt.
        charging_station: Position of the charging station.
        schedule: Cleaning tasks ordered according to their assigned time
            slots by the CSP solver.
        algorithm: Graph-search algorithm to use for navigation.

    Returns:
        complete_path: Combined path followed by the robot, from the charging
            station through all scheduled cleaning locations and back to the
            charging station.
        total_cost: Total navigation cost of the complete journey.
        success: Indicates whether all dirty locations were cleaned and the
            robot returned to the charging station successfully.
    """
    result = run_integration(grid, charging_station, schedule, algorithm)
    return result.complete_path, result.total_cost, result.success


def plan_and_execute(
    grid: Grid, constraints: Constraints, algorithm: str
) -> IntegrationResult:
    """Run the CSP solver, then execute its schedule (spec §7.1, REQ-INT-09).

    Args:
        grid: Environment loaded from Environment.txt.
        constraints: Scheduling problem loaded from Schedule.txt.
        algorithm: Search algorithm name.

    Returns:
        The integration result. If the CSP finds no schedule, no search is
        run and the result reports failure with the robot still at the
        charger (spec §7.1 item 6).
    """
    assignment = backtracking_search(
        list(constraints.variables), initial_domains(constraints), constraints
    )
    if assignment is None:
        return IntegrationResult(
            algorithm,
            [],
            None,
            [],
            [],
            [grid.charger],
            0,
            False,
            "CSP solver found no feasible schedule",
        )
    schedule = schedule_from_assignment(assignment)
    result = run_integration(grid, grid.charger, schedule, algorithm)
    # The dataclass is frozen, so replace() makes a copy with the assignment.
    return replace(result, assignment=assignment)

In [ ]:
for algorithm in ALGORITHMS:
    print(format_result(plan_and_execute(grid, constraints, algorithm), grid))
    print()

## 5. Testing and validation

Three test runs (spec §8.2). Each prints the configuration (the input files and the grid),
the schedule with its constraint check, and the integration output for UCS and A\*. Invalid
input is caught and reported (`InputError`), never a crash.

In [ ]:
def run_case(env_name, schedule_name):
    """One test run: configuration, schedule proof, integration per algorithm.

    Input errors are reported, not raised, so a bad file never stops the
    notebook (spec §8.2 failure case).

    Args:
        env_name: Environment file name.
        schedule_name: Schedule file name.
    """
    print(f"Configuration: {env_name} + {schedule_name}")
    try:
        g = load_environment(find_file(env_name))
        c = load_schedule(find_file(schedule_name), g)
    except (InputError, OSError) as err:
        print(f"FAILURE detected while loading input: {err}")
        return
    print(format_grid(g))
    print(LEGEND)
    print()
    a = backtracking_search(list(c.variables), initial_domains(c), c)
    print(format_schedule_proof(a, c))
    for algorithm in ALGORITHMS:
        print()
        print(format_result(plan_and_execute(g, c, algorithm), g))

### Test run 1: our own environment (`Environment.txt`, `Schedule.txt`)

In [ ]:
run_case(ENV_FILE, SCHEDULE_FILE)

*Screenshot of test run 1 (configuration and output):*

<!-- SCREENSHOT: test run 1 — paste image here -->

### Test run 2: provided sample (`Environment_Sample.txt`, `Schedule_Sample.txt`)

In [ ]:
run_case(SAMPLE_ENV_FILE, SAMPLE_SCHEDULE_FILE)

*Screenshot of test run 2 (configuration and output):*

<!-- SCREENSHOT: test run 2 — paste image here -->

### Test run 3: failure case (`Environment_Error.txt`, `Schedule_Error.txt`)

`Environment_Error.txt` is our environment with two extra obstacles at `(8,0)` and `(9,1)`,
which wall in `D5` at `(9,0)`. The schedule is still feasible, so the CSP succeeds; the
search for the leg to `D5` finds no path, and the system must detect this and report failure
instead of claiming the task was completed.

In [ ]:
run_case(ERROR_ENV_FILE, ERROR_SCHEDULE_FILE)

*Screenshot of test run 3 (failure detected and reported):*

<!-- SCREENSHOT: test run 3 — paste image here -->